# SmartFarm ML — Stage 04: Data reality (cleaning + bootstrap labels)
The make-or-break stage. Live sensors send garbage: sentinel errors (255, -1), impossible
values, and dropped readings (NaN). A model trained on a clean CSV will produce nonsense the
moment a `255` reaches it. So we build a **validation gate** (like `@Valid` at an API boundary)
that runs BEFORE the model, and a **logger** that stamps each reading with the rule's decision
as a bootstrap label — so we grow a labelled dataset even before we have real ground truth.
Data: the messy `irrigation_readings.csv` from Stage 00.

## 1. Reload the messy data and recall the garbage

In [1]:
import pandas as pd, numpy as np

df = pd.read_csv("irrigation_readings.csv", parse_dates=["timestamp"])
print("rows:", len(df))
print("impossible soil (not 0..100):",
      ((df["soil_moisture"] < 0) | (df["soil_moisture"] > 100)).sum())
print("missing values per column:")
print(df.isna().sum())

rows: 660
impossible soil (not 0..100): 10
missing values per column:
timestamp        0
crop_type        0
growth_stage     0
soil_moisture    0
air_humidity     7
temperature      3
irrigated        0
dtype: int64


## 2. Define the contract — valid physical ranges per sensor
This is the single source of truth for "what is a physically possible reading". Anything
outside these is a sensor error, not data. Keep it in one dict so it's easy to audit.

In [2]:
VALID_RANGES = {
    "soil_moisture": (0, 100),   # capacitive sensor, percent
    "air_humidity":  (0, 100),   # DHT11, percent
    "temperature":   (0, 60),    # DHT11, deg C (Sri Lanka field: nothing near 60 is real)
}

## 3. Step 1 — turn impossible values into NaN
We don't silently delete rows yet. First we mark every out-of-range value as NaN (missing),
so sentinels like 255 / -1 join the genuinely-dropped readings in one 'missing' bucket.

In [3]:
clean = df.copy()
for col, (lo, hi) in VALID_RANGES.items():
    bad = (clean[col] < lo) | (clean[col] > hi)   # boolean mask of out-of-range values
    print(f"{col}: {bad.sum()} impossible values -> NaN")
    clean.loc[bad, col] = np.nan                   # set only those cells to NaN

print("\nmissing per column AFTER marking impossibles:")
print(clean[list(VALID_RANGES)].isna().sum())

soil_moisture: 10 impossible values -> NaN
air_humidity: 0 impossible values -> NaN
temperature: 0 impossible values -> NaN

missing per column AFTER marking impossibles:
soil_moisture    10
air_humidity      7
temperature       3
dtype: int64


## 4. Step 2 — handle the gaps
Now every bad cell is NaN. Options: (a) drop the whole row, (b) forward-fill (carry the last
good reading forward). For slow-changing field sensors, a single missing reading is well
approximated by the previous one, so we forward-fill **per crop** (never carry one crop's
value into another). Any NaN still left at the very start (no earlier reading to copy) we drop.

In [8]:
clean = clean.sort_values("timestamp")
sensor_cols = list(VALID_RANGES)

# forward-fill within each crop only (groupby keeps crops separate)
clean[sensor_cols] = clean.groupby("crop_type")[sensor_cols].ffill()

before = len(clean)
clean = clean.dropna(subset=sensor_cols)     # drop leading rows that had nothing to fill
after = len(clean)
print(f"rows dropped (unfillable leading gaps): {before - after} : {before} : {after}")
print(f"clean rows remaining: {after}")
print("missing now:", clean[sensor_cols].isna().sum().to_dict())

rows dropped (unfillable leading gaps): 0 : 660 : 660
clean rows remaining: 660
missing now: {'soil_moisture': 0, 'air_humidity': 0, 'temperature': 0}


## 5. Before vs after — did cleaning actually fix the ranges?
Compare describe() min/max before and after. The 255 / -1 blowouts should be gone.

In [11]:
print("BEFORE  soil min/max:", df["soil_moisture"].min(), df["soil_moisture"].max())
print("AFTER   soil min/max:", clean["soil_moisture"].min(), clean["soil_moisture"].max())

print("______________________")

print("BEFORE  air min/max:", df["air_humidity"].min(), df["air_humidity"].max())
print("AFTER   air min/max:", clean["air_humidity"].min(), clean["air_humidity"].max())

print("______________________")

print("BEFORE  tmp min/max:", df["temperature"].min(), df["temperature"].max())
print("AFTER   tmp min/max:", clean["temperature"].min(), clean["temperature"].max())

BEFORE  soil min/max: -1.0 255.0
AFTER   soil min/max: 35.0 78.5
______________________
BEFORE  air min/max: 48.9 100.0
AFTER   air min/max: 48.9 100.0
______________________
BEFORE  tmp min/max: 17.3 35.2
AFTER   tmp min/max: 17.3 35.2


## 6. The bootstrap logger — grow labelled data with no ground truth
Day one you have no "correct" answers. But you have rules. So: for every incoming reading,
**validate it, and if valid, stamp it with the rule's decision** and append to a log. Weeks
later that log is your first training set. This is the function you'd call inside your
collector / sensor pipeline.

In [6]:
def validate_reading(r: dict):
    """Return a clean reading, or None if it's physically impossible (reject it)."""
    for col, (lo, hi) in VALID_RANGES.items():
        v = r.get(col)
        if v is None or not (lo <= v <= hi):   # missing or out of range -> reject
            return None
    return r

# crop-aware rules baseline = our bootstrap labeller
CROP_IDEAL = {"tomato": 58, "chili": 45, "okra": 50}
def rule_decision(r: dict) -> int:
    return int(r["soil_moisture"] < CROP_IDEAL[r["crop_type"]])

def log_reading(r: dict, log_path="reading_log.csv"):
    clean_r = validate_reading(r)
    if clean_r is None:
        print(f"  REJECTED (sensor garbage): {r}")
        return None
    row = {**clean_r, "rule_label": rule_decision(clean_r)}
    pd.DataFrame([row]).to_csv(
        log_path, mode="a", header=not pd.io.common.file_exists(log_path), index=False)
    print(f"  logged: soil={clean_r['soil_moisture']} crop={clean_r['crop_type']} -> label={row['rule_label']}")
    return row

## 7. Demo the logger on a few live-style readings (one is garbage)

In [7]:
import os
if os.path.exists("reading_log.csv"): os.remove("reading_log.csv")   # fresh start for the demo

incoming = [
    {"crop_type": "tomato", "soil_moisture": 40.0, "air_humidity": 70.0, "temperature": 30.0},
    {"crop_type": "chili",  "soil_moisture": 52.0, "air_humidity": 66.0, "temperature": 28.0},
    {"crop_type": "okra",   "soil_moisture": 255.0,"air_humidity": 60.0, "temperature": 27.0}, # sensor error
    {"crop_type": "tomato", "soil_moisture": 30.0, "air_humidity": 80.0, "temperature": 33.0},
]
for r in incoming:
    log_reading(r)

print("\n--- reading_log.csv contents ---")
print(pd.read_csv("reading_log.csv").to_string())

  logged: soil=40.0 crop=tomato -> label=1
  logged: soil=52.0 crop=chili -> label=0
  REJECTED (sensor garbage): {'crop_type': 'okra', 'soil_moisture': 255.0, 'air_humidity': 60.0, 'temperature': 27.0}
  logged: soil=30.0 crop=tomato -> label=1

--- reading_log.csv contents ---
  crop_type  soil_moisture  air_humidity  temperature  rule_label
0    tomato           40.0          70.0         30.0           1
1     chili           52.0          66.0         28.0           0
2    tomato           30.0          80.0         33.0           1


## Takeaway
- **Clean before the model, always.** A validation gate turns sensor garbage into a controlled
  'missing' state instead of a silent wrong answer. Impossible = reject; missing = fill or drop.
- **Forward-fill per group** — never let one crop's (or device's) reading leak into another.
- **The logger solves the day-one data problem:** no ground truth needed. Each valid reading is
  stored with the rule's decision as a bootstrap label. That log becomes your first training set,
  and later you replace/augment rule-labels with real outcomes (Stage 07).
- This is also why the rules must stay: they are your labeller AND your fail-safe.

## Your turn
1. Add a reading with `temperature = None` (a dropped DHT11 read). Does `validate_reading`
   reject it? Which line catches it?
2. Change the forward-fill to a plain `clean.dropna()` (no fill). How many more rows do you lose?
   When would dropping be safer than filling?
3. (Architecture) In your SmartFarm, where should `validate_reading` live — in sensor-service,
   in the collector before storage, or just before the model call? Why? (hint: the earlier the
   gate, the less garbage spreads.)